In [1]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html
from IPython.display import HTML
import csv

In [2]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

In [3]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [4]:
df = pd.read_csv("A股指数.csv")
a_share_dict = dict(zip(df["Ticker"], df["Name"]))

df = pd.read_csv("海外指数.csv")
other_market_dict = dict(zip(df["Ticker"], df["Name"]))

print(len(a_share_dict), len(other_market_dict))

333 57


In [5]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)

In [6]:
UNIVERSE_A = {}
UNIVERSE_O = {}
target_dates = {}

for ticker in INDEX_DATA:
    if ticker in a_share_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_A[ticker] = INDEX_DATA[ticker]

    elif ticker in other_market_dict:
        target_dates[ticker] = INDEX_DATA[ticker].index[-1]
        UNIVERSE_O[ticker] = INDEX_DATA[ticker]

In [7]:
print(f"Assets under Analysis: {len(UNIVERSE_A)}  {len(UNIVERSE_O)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 73  35
Total Assets: 186


In [8]:
n = 0
for ticker in other_market_dict:
    if ticker in INDEX_DATA:
        n += 1

n

39

<br><br><br><br>
## 多投列表

In [12]:
UNIVERSE = UNIVERSE_A
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 3
duration = int(years * 252)

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [13]:
display_number = 15

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

In [14]:
print()
print("A股多投")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


A股多投
2025-06-16
3年


,Ticker,指数名称,F1
64,399966.SZ,中证800证保,-1.622480
41,h30035.CSI,300非银,-1.549778
11,399812.SZ,养老产业,-1.498982
51,931139.CSI,CS消费50,-1.497883
53,931068.CSI,消费龙头,-1.417517
12,000815.CSI,细分食品,-1.184763
56,930653.CSI,CS食品饮,-1.179976
8,707717L.MI,MSCI中国A股质量,-0.814344
1,399814.SZ,农业,-0.657251
5,399975.SZ,证券,-0.488625


In [18]:
UNIVERSE = UNIVERSE_O
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 3
duration = int(years * 252)

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [19]:
display_number = 15

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

In [20]:
print()
print("海外多投")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


海外多投
2025-06-16
3年


,Ticker,指数名称,F1
26,931454.CSI,港股消费易方达,-1.051905
29,987008.CNI,港股通科技30,-0.878922
24,930604.CSI,中概互联,-0.834415
4,HSTECH.HI,恒生科技基金,-0.817893
10,931787CNY00.CSI,港股创新药,-0.752026
16,HSSCNE.HI,恒生新经济,-0.704849
8,HXC.GI,纳斯达克中国金龙,-0.654197
25,931024.CSI,港股非银,-0.567776
15,HSCGSI.HI,恒生消费,-0.491473
32,TPX.GI,日本东证指数,-0.362894


<br><br><br><br>
## 空投列表

In [24]:
UNIVERSE = UNIVERSE_A
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 3
duration = int(years * 252)

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [25]:
display_number = 15

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=False)  # previously True
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=True)  # previously False

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

In [26]:
print()
print("A股空投")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


A股空投
2025-06-16
3年


,Ticker,指数名称,F1
26,931151.CSI,光伏产业,19.809228
63,399967.SZ,中证军工,3.722816
32,931409.CSI,SHS创新药,3.372957
31,930902.CSI,中证数据,2.699244
13,000814.SH,细分医药,2.580037
9,HSSSHID.HI,恒生创新药50,2.373937
39,h30202.CSI,软件指数,2.325977
29,931152.CSI,CS创新药,2.118946
4,830009.XI,富时中国A50,2.102800
66,000934.SH,800金地,2.100198


In [30]:
UNIVERSE = UNIVERSE_O
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 3
duration = int(years * 252)

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    sample_data.dropna(inplace = True)
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [31]:
display_number = 15

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', 'F1'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', 'F2'])

# Sort the dataframes in reverse order
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='F1', ascending=False)  # Changed to descending
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='F2', ascending=True)  # Changed to ascending

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

# Map index names and select columns
top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "F1"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "F2"]]

In [32]:
print()
print("海外空投")
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


海外空投
2025-06-16
3年


,Ticker,指数名称,F1
30,h11146.CSI,港股通金融,3.257250
31,h30106.CSI,港股金融,3.222720
13,122489.MI,MSCI金龙,2.944938
28,931790.CSI,中韩半导体,2.185403
18,HSCEI.HI,恒生中国企业,2.142256
6,GDAXI.GI,德国,2.018449
17,HSBBAC.HI,恒生大湾区,2.002258
2,HSI.HI,恒生,1.933872
19,SPHCMSHP.SPI,标普香港中小盘,1.894406
9,FCHI.GI,法国CAC40,1.459876
